# Task 2 — Validation of Temporal Knowledge Graphs (Google Colab / Jupyter)

Этот блокнот является **естественным продолжением Task 1**.

Эксперт загружает YAML-файл, созданный в первом блокноте, после чего блокнот автоматически:
- строит **эталонный (gold) граф**, который точно воспроизводит ручную reasoning-схему из YAML;
- строит **автоматический temporal KG** по тем же статьям через пайплайн репозитория;
- сохраняет обе версии графа, таблицы триплетов, comparison summary и визуализации;
- показывает эксперту **текстовое представление триплетов** и **интерактивные визуализации** для ручной валидации.


# Пошаговый туториал

## Шаг 1. Запустите ячейку «Установка и импорт»
Она установит зависимости для визуализации и подключит функции репозитория.

## Шаг 2. Запустите ячейку «Вспомогательные функции»
Она подготовит UI, рендеринг таблиц и графов.

## Шаг 3. Запустите ячейку «Форма Task 2»
В ней нужно:
1. загрузить YAML из Task 1;
2. выбрать, строить ли автоматический граф;
3. нажать кнопку запуска.

## Что вы получите
- **Gold graph**: полностью повторяет reasoning trajectory эксперта;
- **Auto graph**: строится одной командой поверх статей из YAML;
- **Triplets tables**: CSV/JSONL для ручной проверки;
- **Comparison summary**: базовое сравнение gold vs auto;
- **Reference scout**: дополнительные ссылки-кандидаты для примеров/контрпримеров.

## Новые temporal-поля в Task 2
- `start_date` / `end_date` — окно наблюдения/assertion extraction;
- `valid_from` / `valid_to` — окно валидности самого открытия;
- в обоих слоях допустимы `-infinity` и `+infinity`.


In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
!pip -q install ipywidgets pyyaml requests pandas panel hvplot pyvis
!pip -q install -e .[temporal,notebook_viz]

import json, os, tempfile
from pathlib import Path

import pandas as pd
import ipywidgets as W
from IPython.display import display, Markdown, HTML, IFrame, clear_output

try:
    import panel as pn
    pn.extension()
except Exception:
    pn = None

try:
    import hvplot  # noqa: F401
    import hvplot.networkx  # noqa: F401
except Exception:
    pass

from scireason.task2_validation import (
    build_task2_validation_bundle,
    load_task1_yaml,
    make_hvplot_payload,
)

print("Готово. Теперь запускайте следующую ячейку: «Вспомогательные функции»." )


In [ ]:
# @title
# Вспомогательные функции (не нужно редактировать)

def _save_uploaded_yaml(upload_value, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        # legacy ipywidgets
        name, meta = next(iter(upload_value.items()))
        content = meta["content"]
        path = out_dir / name
        path.write_bytes(content)
        return path
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        name = meta.get("name") or "task1.yaml"
        content = meta.get("content")
        path = out_dir / name
        path.write_bytes(bytes(content))
        return path
    raise ValueError("Не удалось прочитать загруженный файл. Загрузите YAML ещё раз.")


def _display_manifest(manifest_path: Path):
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    lines = [
        f"- **topic**: {manifest.get('topic')}",
        f"- **bundle_dir**: `{manifest.get('bundle_dir')}`",
        f"- **gold_graph**: `{manifest.get('gold_graph')}`",
        f"- **gold_triplets_csv**: `{manifest.get('gold_triplets_csv')}`",
    ]
    if manifest.get("auto_run_dir"):
        lines += [
            f"- **auto_run_dir**: `{manifest.get('auto_run_dir')}`",
            f"- **auto_graph_json**: `{manifest.get('auto_graph_json')}`",
            f"- **auto_triplets_csv**: `{manifest.get('auto_triplets_csv')}`",
        ]
    if manifest.get("comparison_summary"):
        lines.append(f"- **comparison_summary**: `{manifest.get('comparison_summary')}`")
    if manifest.get("reference_scout"):
        lines.append(f"- **reference_scout**: `{manifest.get('reference_scout')}`")
    display(Markdown("## Артефакты Task 2
" + "
".join(lines)))
    return manifest


def _show_dataframe(csv_path: str, title: str, max_rows: int = 100):
    p = Path(csv_path)
    if not p.exists():
        display(Markdown(f"**{title}:** файл не найден"))
        return
    df = pd.read_csv(p)
    display(Markdown(f"## {title} ({len(df)} строк)"))
    display(df.head(max_rows))


def _show_graph(graph_json_path: str, html_path: str, title: str):
    display(Markdown(f"## {title}"))
    gj = Path(graph_json_path)
    hp = Path(html_path)
    if gj.exists():
        try:
            payload = json.loads(gj.read_text(encoding="utf-8"))
            try:
                _, plot = make_hvplot_payload(payload)
                display(plot)
            except Exception as e:
                display(Markdown(f"hvPlot не удалось построить: `{e}`"))
        except Exception as e:
            display(Markdown(f"Не удалось открыть JSON графа: `{e}`"))
    if hp.exists():
        try:
            display(IFrame(src=str(hp), width="100%", height=760))
        except Exception:
            display(HTML(f'<a href="{hp}">{hp.name}</a>'))


In [ ]:
# @title
# Форма Task 2 (запустите ячейку и заполните поля)

yaml_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Загрузить YAML')
out_dir = W.Text(value='runs/task2_validation', description='Out dir', layout=W.Layout(width='100%'))
run_auto = W.Checkbox(value=True, description='Строить автоматический temporal KG')
reference_scout = W.Checkbox(value=True, description='Сгенерировать reference scout')
multimodal = W.Checkbox(value=True, description='Использовать multimodal pipeline')
run_btn = W.Button(description='Построить Task 2 bundle', button_style='success')
status = W.HTML()
out = W.Output()


def _on_run(_):
    with out:
        clear_output()
        try:
            status.value = '<b>Запуск...</b>'
            workdir = Path(tempfile.mkdtemp(prefix='task2_notebook_'))
            trajectory_path = _save_uploaded_yaml(yaml_upload.value, workdir)
            task1 = load_task1_yaml(trajectory_path)
            display(Markdown(f"# Загружен YAML
- **topic**: {task1.get('topic')}
- **submission_id**: `{task1.get('submission_id')}`"))

            bundle = build_task2_validation_bundle(
                trajectory_path,
                out_dir=Path(out_dir.value),
                include_auto_pipeline=bool(run_auto.value),
                multimodal=bool(multimodal.value),
                enable_reference_scout=bool(reference_scout.value),
            )
            manifest = _display_manifest(bundle.manifest_path)

            _show_dataframe(manifest['gold_triplets_csv'], 'Gold triplets')
            _show_graph(manifest['gold_graph'], manifest['gold_graph_html'], 'Gold graph')

            if manifest.get('auto_triplets_csv'):
                _show_dataframe(manifest['auto_triplets_csv'], 'Auto triplets')
            if manifest.get('auto_graph_json') and manifest.get('auto_graph_html'):
                _show_graph(manifest['auto_graph_json'], manifest['auto_graph_html'], 'Auto graph')
            if manifest.get('comparison_summary'):
                cmp = json.loads(Path(manifest['comparison_summary']).read_text(encoding='utf-8'))
                display(Markdown('## Comparison summary'))
                display(pd.DataFrame([cmp]))
            if manifest.get('reference_scout'):
                scout = json.loads(Path(manifest['reference_scout']).read_text(encoding='utf-8'))
                rows = []
                for hit in scout.get('hits', []):
                    for res in hit.get('results', []):
                        rows.append({
                            'query': hit.get('query'),
                            'id': res.get('id'),
                            'title': res.get('title'),
                            'year': res.get('year'),
                            'url': res.get('url'),
                        })
                if rows:
                    display(Markdown('## Reference scout'))
                    display(pd.DataFrame(rows).head(100))
            status.value = '<span style="color:green"><b>Готово.</b></span>'
        except Exception as e:
            status.value = f'<span style="color:red"><b>Ошибка:</b> {e}</span>'
            raise

run_btn.on_click(_on_run)

display(W.VBox([
    W.HTML('<h2>Task 2 — Validation UI</h2>'),
    yaml_upload,
    out_dir,
    run_auto,
    multimodal,
    reference_scout,
    run_btn,
    status,
    out,
]))


# CLI-режим (тот же workflow без notebook)

Если удобнее запускать всё одной командой из терминала, используйте:

```bash
top-papers-graph task2-bundle   --trajectory path/to/task1.yaml   --out-dir runs/task2_validation   --multimodal   --reference-scout
```

Эта команда создаёт тот же набор артефактов, который показывает блокнот.


# FAQ

## Что считается эталонным графом?
Эталонный граф строится **только из YAML Task 1** и полностью повторяет шаги эксперта, их evidence и связи между шагами.

## Что считается автоматическим графом?
Автоматический граф строится из **списка статей в YAML** и запускает встроенный конвейер репозитория: статьи → ingestion → temporal KG → review assets → report.

## Что делать, если автоматический граф почти пустой?
Это ожидаемо, если PDF недоступны, в статьях мало структурируемого текста или temporal evidence неявен. В этом случае ориентируйтесь на:
- `reference_scout.json`
- triplets CSV
- `comparison_summary.json`
- review_queue из auto pipeline

## Как интерпретировать temporal поля?
- `start_date` / `end_date` — когда связь наблюдается / извлечена;
- `valid_from` / `valid_to` — когда открытие считается валидным;
- `temporal_basis` — откуда взялось время (publication date, publication year proxy, manual YAML source и т.д.).
